# Самостоятельная работа: от разрозненных выгрузок к витрине активности пользователей

Одна строка `users.csv` описывает пользователя. Таблица `monthly_activity_wide.csv` хранит показатели за три месяца в отдельных столбцах. Ваша задача — преобразовать данные в витрину уровня «пользователь × месяц», сравнить методы масштабирования и применить готовый preprocessing к новой выгрузке.

Полное условие находится в `assignment_brief.md`, словарь полей — в `data_dictionary.md`.

Все программные ячейки снабжены русскими комментариями и подсказками. Пустые места отмечены `TODO`. Notebook можно запустить сверху вниз до начала работы: незаполненные этапы будут отмечены сообщением, а не ошибкой.

## 1. Подготовка окружения

In [1]:
# Базовые библиотеки для работы с таблицами, расчётами и графиками.
from pathlib import Path
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Функция ищет корень проекта независимо от того, где запущен notebook:
# из корня проекта, из папки notebooks, из VS Code или из Google Colab.
def find_project_root(required_files=("users.csv", "monthly_activity_wide.csv", "tariffs.csv", "new_users.csv")):
    candidates = []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])

    # В Google Colab рабочая директория обычно находится внутри /content.
    content_dir = Path("/content")
    if content_dir.exists():
        candidates.append(content_dir)
        # Ищем вложенный проект после распаковки ZIP-архива.
        candidates.extend(path.parent.parent.parent for path in content_dir.glob("**/data/raw/users.csv"))

    checked = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.add(candidate)
        raw_dir = candidate / "data" / "raw"
        if all((raw_dir / filename).exists() for filename in required_files):
            return candidate

    raise FileNotFoundError(
        "Не найден каталог data/raw с учебными файлами. "
        "Проверьте, что архив распакован полностью и структура папок сохранена."
    )

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Создаём каталоги результатов, если они ещё не существуют.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", PROJECT_ROOT)
print("Исходные данные:", RAW_DIR)
print("Обработанные данные:", PROCESSED_DIR)
print("Результаты и графики:", OUTPUT_DIR)

Корень проекта: /mnt/data/user_activity_homework_transform_scaling
Исходные данные: /mnt/data/user_activity_homework_transform_scaling/data/raw
Обработанные данные: /mnt/data/user_activity_homework_transform_scaling/data/processed
Результаты и графики: /mnt/data/user_activity_homework_transform_scaling/outputs


In [2]:
# Универсальная функция преобразует денежные строки вида "1 234,50 ₽" в float.
def parse_money(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.replace("₽", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")

# Функция приводит текстовые категории к нижнему регистру и убирает лишние пробелы.
def normalize_text(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower()

# В разных версиях scikit-learn параметр плотного вывода называется по-разному.
# Эта функция сохраняет совместимость со старыми и новыми версиями библиотеки.
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

## 2. Загрузка данных

In [3]:
# Этот стартовый блок уже готов: он загружает исходные файлы.
users_raw = pd.read_csv(RAW_DIR / "users.csv")
activity_raw = pd.read_csv(RAW_DIR / "monthly_activity_wide.csv")
tariffs_raw = pd.read_csv(RAW_DIR / "tariffs.csv")
new_users_raw = pd.read_csv(RAW_DIR / "new_users.csv")

print("users.csv:", users_raw.shape)
print("monthly_activity_wide.csv:", activity_raw.shape)
print("tariffs.csv:", tariffs_raw.shape)
print("new_users.csv:", new_users_raw.shape)

display(users_raw.head())
display(activity_raw.head(3))

users.csv: (425, 5)
monthly_activity_wide.csv: (423, 16)
tariffs.csv: (5, 5)
new_users.csv: (36, 17)


,user_id,registration_date,region,segment,tariff_id
0,U0001,2024-10-08,поволжье,Массовый,T02
1,U0002,2024-04-11,МОСКВА,МАССОВЫЙ,T01
2,U0003,2023-10-10,Поволжье,Массовый,T01
3,U0004,2023-09-12,санкт-петербург,активный,T02
4,U0005,2024-02-22,санкт-петербург,Активный,T03


,user_id,sessions_2026_01,minutes_2026_01,spend_2026_01,support_calls_2026_01,satisfaction_2026_01,sessions_2026_02,minutes_2026_02,spend_2026_02,support_calls_2026_02,satisfaction_2026_02,sessions_2026_03,minutes_2026_03,spend_2026_03,support_calls_2026_03,satisfaction_2026_03
0,U0001,27.0,1362.5,"824,48 ₽",1.0,"4,40",26.0,970.1,"648,26 ₽",1.0,"4,29",32.0,1304.9,"832,56 ₽",NaN,"4,53"
1,U0002,27.0,1338.1,"531,94 ₽",0.0,"4,70",40.0,2068.3,"757,63 ₽",1.0,"4,28",44.0,2000.7,"825,81 ₽",2.0,"3,59"
2,U0003,20.0,758.6,"437,32 ₽",0.0,"5,00",20.0,742.3,"456,31 ₽",2.0,"4,58",27.0,1508.9,"617,55 ₽",3.0,"3,30"


### Задание 1. Диагностика качества

Создайте таблицу, где для каждого файла указаны число строк, число столбцов, число пропусков и число полных дубликатов. Сохраните её в `outputs/data_quality_report.csv`.

In [4]:
# TODO: создайте функцию quality_summary и соберите quality_report.
# Подсказка: используйте len(df), df.shape[1], df.isna().sum().sum(), df.duplicated().sum().

print("TODO: выполните диагностику четырёх таблиц.")

TODO: выполните диагностику четырёх таблиц.


## 3. Преобразование пользователей и тарифов

In [5]:
# TODO 1: создайте users = users_raw.copy().
# TODO 2: преобразуйте registration_date через pd.to_datetime(..., errors="coerce").
# TODO 3: нормализуйте region и segment функцией normalize_text.
# TODO 4: объедините мск с москва, а спб с санкт-петербург.
# TODO 5: удалите дубликаты по user_id.
# TODO 6: подготовьте tariffs и преобразуйте monthly_fee функцией parse_money.

print("TODO: подготовьте таблицы users и tariffs.")

TODO: подготовьте таблицы users и tariffs.


## 4. Широкий и длинный формат

In [6]:
# TODO 1: удалите полные дубликаты activity_raw.
# TODO 2: выполните melt, сохранив user_id как идентификатор.
# TODO 3: разделите metric_month на metric и month.
# TODO 4: преобразуйте raw_value в число с учётом показателя.
# TODO 5: соберите activity_long через pivot_table.
# TODO 6: проверьте уникальность пары user_id + month.
# TODO 7: сохраните user_activity_long.csv.

print("TODO: сформируйте activity_melted и activity_long.")

TODO: сформируйте activity_melted и activity_long.


In [7]:
# Необязательная самопроверка после выполнения предыдущего блока.
if "activity_long" in globals():
    print("Размер activity_long:", activity_long.shape)
    print("Дубликатов user_id + month:", activity_long.duplicated(["user_id", "month"]).sum())
    display(activity_long.head())
else:
    print("Шаг ожидает выполнения: переменная activity_long пока не создана.")

Шаг ожидает выполнения: переменная activity_long пока не создана.


## 5. Объединение и признаки

In [8]:
# TODO 1: соедините activity_long с users через left merge и validate="many_to_one".
# TODO 2: добавьте tariffs с тем же контролем кардинальности.
# TODO 3: создайте month_date.
# TODO 4: рассчитайте семь обязательных производных признаков.
# TODO 5: сохраните user_activity_datamart.csv.

print("TODO: соберите datamart и создайте признаки.")

TODO: соберите datamart и создайте признаки.


## 6. Агрегированная витрина

In [9]:
# TODO: сгруппируйте datamart по segment и tariff_name.
# Рассчитайте число уникальных пользователей, средние sessions, minutes, spend,
# satisfaction, support_calls и долю low_satisfaction.
# Сохраните segment_tariff_summary.csv.

print("TODO: создайте segment_tariff_summary.")

TODO: создайте segment_tariff_summary.


## 7. Графическая проверка

In [10]:
# TODO: постройте и сохраните минимум четыре графика.
# 1. Гистограмма spend.
# 2. Boxplot spend по tariff_name.
# 3. Среднее sessions по тарифам.
# 4. Уникальные пользователи по segment.
# У каждого графика должны быть заголовок и подписи осей.

print("TODO: постройте графики и сохраните PNG в outputs.")

TODO: постройте графики и сохраните PNG в outputs.


## 8. Масштабирование

In [11]:
# TODO 1: выберите минимум пять количественных признаков.
# TODO 2: заполните числовые пропуски медианой.
# TODO 3: вручную рассчитайте Min-Max и z-score для одного значения spend.
# TODO 4: примените MinMaxScaler, StandardScaler и RobustScaler.
# TODO 5: сравните статистики и сохраните scaling_comparison.csv.
# TODO 6: сохраните scaled_features.csv.

print("TODO: сравните три метода масштабирования.")

TODO: сравните три метода масштабирования.


## 9. Preprocessing новой выгрузки

In [12]:
# TODO 1: определите numeric_features и categorical_features.
# TODO 2: соберите numeric_pipeline: SimpleImputer + выбранный scaler.
# TODO 3: соберите categorical_pipeline: SimpleImputer + OneHotEncoder.
# TODO 4: объедините контуры через ColumnTransformer.
# TODO 5: выполните fit только на исторической витрине.
# TODO 6: примените transform к new_users_raw.
# TODO 7: сохраните new_users_preprocessed.csv и preprocessor.joblib.

print("TODO: соберите воспроизводимый preprocessing.")

TODO: соберите воспроизводимый preprocessing.


## 10. Итоговый вывод

Напишите 8–12 предложений по структуре:

1. Что было преобразовано.
2. Какая гранулярность получилась.
3. Какие проблемы качества были найдены.
4. Какие различия есть между тарифами или сегментами.
5. Как выбросы повлияли на scaler-методы.
6. Какой scaler выбран и почему.
7. Как новая выгрузка проходит через `transform`.
8. Какое ограничение есть у анализа.

**Ваш вывод:**

TODO: напишите вывод своими словами и приведите конкретные значения.

## 11. Финальная самопроверка

Откройте `self_checklist.md` и убедитесь, что все обязательные результаты созданы. Затем перезапустите среду и выполните notebook сверху вниз.